# ラビ振動

このノートブックでは、 Qubex のシミュレータ機能を使ってラビ振動の実験を行う方法を示します。

## 背景

ラビ振動の数理的な背景について簡潔にまとめます。
詳細を知りたい場合は [Wikipedia](https://en.wikipedia.org/wiki/Rabi_cycle) 等を参照してください。

ラビ振動は二準位系 $H=\frac{\Delta}{2}Z + \frac{\Omega}{2}X$ に対して定義される現象です。 
$\Delta$ は detuning といい、 $\Omega$ はラビ周波数といいます。
このハミルトニアンにおいて初期状態が $\ket{0}$ の場合、 $t$ 秒後に $\ket{1}$ となる確率 $P_1(t)$ は次のように表せます。

$$
P_1(t) = \frac{\Omega^2}{\Omega^2 + \Delta^2} \sin^2( \frac{\sqrt{\Omega^2 + \Delta^2}}{2} t )
$$

$\Delta=0$ の場合は共鳴ラビ振動といい、 $\ket{1}$ となる確率が 0 と 1 の間で振動します。
一方、 $\Delta\neq 0$ の場合は非共鳴ラビ振動といい、 $\ket{1}$ となる確率は 0 と 1 未満の間で振動します。

### トランズモンを外部マイクロ波でドライブする

トランズモンのハミルトニアンは周波数 $\omega_q$ と非調和度 $\alpha$ を次のように表せます。

$$
H_0 = \omega_q a^{\dagger} a + \frac{\alpha}{2} a^{\dagger} a^{\dagger} a a
$$

次の外部マイクロ波 (周波数 $\omega_d$, 振幅 $A(t)$) でドライブします。

$$
H_{\text{drive}}(t) = A(t) \cos (\omega_d t + \phi) (a + a^\dagger)
$$

ドライブ周波数 $\omega_d$ の回転波フレームに移り、回転波近似 (RWA) を行うと、実効ハミルトニアンは次のようになり、ラビ振動が起きます。

$$
H_{\text{eff}}(t) = \frac{\Delta}{2} Z + \frac{\Omega(t)}{2} X
$$

<details>

<summary>導出</summary>

トランズモンを二準位系で近似すると、次のようになります。 ($\Omega(t) = A(t)$)

$$
\begin{align*}
H_0                 &= \frac{\omega_q}{2} Z \\
H_{\text{drive}}(t) &= \Omega(t) \cos (\omega_d t) X \\
H(t)                &= \frac{\omega_q}{2} Z + \Omega(t) \cos (\omega_d t) X
\end{align*}
$$

回転フレームに移行します。
すなわち $U(t) = \exp(- i \frac{\omega_d t}{2} Z)$ でハミルトニアンを変換します。

$$
H^{\prime}    = U^\dagger H U - i U^\dagger \frac{dU}{dt}
$$

このハミルトニアンを計算します。

#### (1) $\frac{\omega_q}{2} Z $ の変換

$$
U^\dagger \frac{\omega_q}{2} Z U = \frac{\omega_q}{2} Z
$$

#### (2) 追加項の計算

$$
- i U^\dagger \frac{dU}{dt} = -\frac{\omega_d}{2} Z
$$

(1) と (2) を合わせると detuning ($\Delta = \omega_q - \omega_d$) の項があらわれます。

$$
\frac{\omega_q}{2} Z - \frac{\omega_d}{2} Z = \frac{\Delta}{2} Z
$$

#### (3) ドライブ項 ($\Omega(t) \cos (\omega_d t) X$) の変換

$$
U^\dagger \Omega(t) \cos (\omega_d t) X U = \Omega(t) \cos(\omega_d t) (X \cos(\omega_d t) + Y \sin(\omega_d t))
$$

なので、三角関数を計算して次のようになります。

$$
H^{\prime}_{\text{drive}}(t) = \frac{\Omega(t)}{2} X + \frac{\Omega(t)}{2} (X \cos(2 \omega_d t) + Y \sin(2 \omega_d t))
$$

回転波近似を行い、 $\cos(2 \omega_d t), \sin(2 \omega_d t)$ といった項を削除します。

よって最終的に次のハミルトニアンが得られ、ラビ振動となります。

$$
H_{\text{eff}}(t) = \frac{\Delta}{2} Z + \frac{\Omega(t)}{2} X
$$

</details>

## Qubex でのトランズモンの扱い

Qubex では、トランズモンを作成するときに以下のパラメータを指定できます。

- label : ラベル
- dimension : ヒルベルト空間の次元
- frequency : 基本遷移周波数 (下記ハミルトニアンの $\omega_{q}$)
- anharmonicity : 非調和度 (下記ハミルトニアンの $\alpha$)
- relaxation_rate : エネルギー緩和率 ($T_1$)
    - 対応する Lindblad 演算子 $L_1 = \sqrt{\gamma_1} a$
- dephasing rate : 位相緩和率 ($T_\varphi$)
    - 対応する Lindblad 演算子 $L_\varphi = \sqrt{\gamma_\varphi} a^\dagger a$

$$
H_0 = \omega_q a^{\dagger} a + \frac{\alpha}{2} a^{\dagger} a^{\dagger} a a
$$

In [1]:
# Install qubex library if not installed
# !pip install git+https://github.com/amachino/qubex.git

In [2]:
import numpy as np

import qubex as qx
from qubex.simulator import Control, QuantumSimulator, QuantumSystem, Transmon

Qubex のシミュレータ機能 (`QuantumSimulator`) を使う場合、シミュレート対象の `QuantumSystem` を最初に作成する必要があります。
`QuantumSystem` は `Object` と `Coupling` から構成されます。
１量子ビットのシミュレートを行う場合は単一の `Object` を与えれば良いです。
今回、トランズモンのラビ振動を行うため、 `Object` には `Transmon` クラスのインスタンスを渡します。

In [3]:
# Define the transmon qubit (unit: GHz, ns)
qubit = Transmon(
    label="Q01",
    dimension=3,
    frequency=7.648,
    anharmonicity=-0.333,
    relaxation_rate=0.00005,
    dephasing_rate=0.00005,
)

# Define the quantum system with the qubit
system = QuantumSystem(objects=[qubit])

# Define the quantum simulator with the system
simulator = QuantumSimulator(system)

トランズモンのハミルトニアンを確認します。
今回トランズモンを `dimension=3` と三準位で近似したため、 $3\times 3$ 行列となります。
非調和度が小さいと仮定すると固有エネルギーは次のように近似できます。

$$
E_n = \omega_q n + \frac{\alpha}{2} n (n-1)
$$

よって、ハミルトニアンは次のようになります。

$$
\begin{align*}
H_{00} &= E_0 = 0 \\
H_{11} &= E_1 = \omega_q = 7.648 \times 2 \pi \approx 48.054 \\
H_{22} &= E_2 = 2 \omega_q + \alpha = (2 \times 7.648 - 0.333) \times 2 \pi \approx 94.015
\end{align*}
$$

Qubex では周波数を GHz で指定しますが、角周波数に変換するために $2 \pi$ をかけています。

In [4]:
# Check the Hamiltonian of the system
system.hamiltonian

Quantum object: dims=[[3], [3]], shape=(3, 3), type='oper', dtype=CSR, isherm=True
Qobj data =
[[ 0.          0.          0.        ]
 [ 0.         48.05380123  0.        ]
 [ 0.          0.         94.01530175]]

`qubex.pulse.Rect` で振幅 $A=\frac{4\pi}{100}$ で時間 100 ns の長方形パルスを作成します。
ラビ周波数は $A/2\pi=0.02 \text{GHz}$ となります。
この振幅で 100 ns 駆動すると、ブロッホ球をおよそ 2 回転します。

In [5]:
# Define the drive pulse

# Rectangular pulse with amplitude 4π/100 and duration 100 ns
# Rabi frequency will be 2/100 = 0.02 GHz = 20 MHz
duration = 100
drive = qx.pulse.Rect(
    duration=duration,
    amplitude=2 * (2 * np.pi) / duration,
)

# Plot the drive pulse
drive.plot()

## 共鳴ラビ振動

ドライブパルスは `Control` クラスで作成します。
ドライブパルスの周波数は明示的に `frequency` という引数で指定できるが、何も渡さない場合は `target` の周波数となります。
共鳴ラビ振動はドライブパルスの周波数とトランズモンの周波数が等しい場合に起きるので、今回は次のように書けます。

エネルギー緩和率や位相緩和率まで考慮した時間発展をシミュレートする場合は `QuantumSimulator` クラスの `mesolve` メソッドを使用します。
次の Lindblad 方程式に基づきシミュレートします。

$$
\frac{d\rho}{dt} = -i [H(t), \rho] + \sum_k (L_k \rho L_k^\dagger - \frac{1}{2} \{ L_k^\dagger L_k, \rho \})
$$

In [6]:
# Define the control with the target qubit and the drive pulse
control = Control(
    target=qubit,
    waveform=drive,
)

# Run the simulation by solving the master equation
result = simulator.mesolve(
    controls=[control],  # List of controls
    initial_state={"Q01": "0"},  # Initial states of the qubits
    n_samples=101,  # Number of samples
)

実行結果は `SimulationResult` として取得できます。
このクラスには様々な可視化メソッドがあるため、これらを用いて、 100 ns で約 2 周期のラビ振動が確認できます。

In [7]:
# Show the last population of the qubit
result.show_last_population(qubit.label)

# Plot the population dynamics of the qubit
result.plot_population_dynamics(qubit.label)

# Plot the Bloch vectors of the qubit
result.plot_bloch_vectors(qubit.label)

# Display the Bloch sphere of the qubit
result.display_bloch_sphere(qubit.label)

<IPython.core.display.Javascript object>

## 非共鳴ラビ振動

ドライブパルスの周波数をトランズモンの周波数とわずかに異なる (detuning, $\Delta=0.01$) ようにして、シミュレートします。
`frame="drive"` と指定することで、ドライブパルスの周波数の回転フレームで結果を表示できます。
$\ket{0}$ と $\ket{1}$ の振動の周波数は $\sqrt{\Omega^2+\Delta^2}$ と共鳴ラビ振動の場合よりも大きいため、このフレームでは2周とすこし回っていることが確認できます。

In [8]:
detuning = 0.01

control = Control(
    target=qubit,
    frequency=qubit.frequency + detuning,
    waveform=drive,
)

result = simulator.mesolve(
    controls=[control],
    initial_state={"Q01": "0"},
    n_samples=101,
)

result.show_last_population(qubit.label)
result.plot_population_dynamics(qubit.label)

# drive frame
result.plot_bloch_vectors(qubit.label, frame="drive")
result.display_bloch_sphere(qubit.label, frame="drive")


<IPython.core.display.Javascript object>

トランズモンの回転フレームで表示すると次のようになります。

In [9]:
# qubit frame
result.plot_bloch_vectors(qubit.label, frame="qubit")
result.display_bloch_sphere(qubit.label, frame="qubit")

<IPython.core.display.Javascript object>